In [28]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report

def time_to_seconds(tstr):
    if pd.isnull(tstr): return 0
    parts = str(tstr).split(':')
    if len(parts) == 3:
        h, m, s = int(parts[0]), int(parts[1]), float(parts[2])
        return h * 3600 + m * 60 + s
    elif len(parts) == 2:
        m, s = int(parts[0]), float(parts[1])
        return m * 60 + s
    else:
        try:
            return float(parts[0])
        except:
            return 0

# 1. 讀檔
winner = pd.read_csv('./data/winners.csv')
drivers = pd.read_csv('./data/drivers_updated.csv')
teams = pd.read_csv('./data/teams_updated.csv')
laps = pd.read_csv('./data/fastest_laps_updated.csv')

# 2. 新增 year 欄（取自 Date）
winner['year'] = pd.to_datetime(winner['Date']).dt.year
winner['year_raw'] = winner['year']
winner['Grand Prix raw'] = winner['Grand Prix']

# 3. 合併 drivers
df = winner.merge(
    drivers[['Driver', 'Car', 'year', 'Nationality', 'PTS']],
    left_on=['Winner', 'Car', 'year'],
    right_on=['Driver', 'Car', 'year'],
    how='left',
    suffixes=('', '_driver')
)

# 4. 合併 fastest_laps
df = df.merge(
    laps[['Grand Prix', 'Driver', 'Car', 'year', 'Time']],
    left_on=['Grand Prix', 'Winner', 'Car', 'year'],
    right_on=['Grand Prix', 'Driver', 'Car', 'year'],
    how='left',
    suffixes=('', '_lap')
)

# 5. 合併 teams
df = df.merge(
    teams[['Team', 'PTS', 'year']],
    left_on=['Car', 'year'],
    right_on=['Team', 'year'],
    how='left',
    suffixes=('', '_team')
)

df['RaceTime_sec'] = df['Time'].apply(time_to_seconds)
df['FastestLap_sec'] = df['Time_lap'].apply(time_to_seconds)

df['year_raw'] = winner['year_raw']
df['Grand Prix raw'] = winner['Grand Prix raw']

# ====== 不再將 Winner 放入 feature，只當 y ======
cat_cols = ['Car', 'Grand Prix', 'Nationality', 'Team']  # Winner 移除
num_cols = [
    'Laps',
    'PTS',
    'PTS_team',
    'RaceTime_sec',
    'FastestLap_sec',
    'year'
]

df['PTS_team'] = df['PTS_team'].fillna(0)
df[num_cols] = df[num_cols].fillna(0)

# 保留只出現2次以上的 Winner
value_counts = df['Winner'].value_counts()
valid_drivers = value_counts[value_counts >= 2].index
df = df[df['Winner'].isin(valid_drivers)]

print("過濾前總樣本數：", winner.shape[0])
print("只出現過一次的車手數量：", sum(value_counts == 1))
print("保留後樣本數：", df.shape[0])

# === LabelEncode features ===
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# === LabelEncode Winner (目標) ===
le_winner = LabelEncoder()
df['Winner_enc'] = le_winner.fit_transform(df['Winner'].astype(str))

# 數值標準化
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# ====== Dataset 準備 ======
X_cat = df[cat_cols].values
X_num = df[num_cols].values
y = df['Winner_enc'].values
num_classes = len(np.unique(y))
print('y範圍:', y.min(), y.max(), 'num_classes:', num_classes)

df = df.reset_index(drop=True)
df['orig_index'] = df.index

# train/test split
X_cat_train, X_cat_test, X_num_train, X_num_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_cat, X_num, y, df['orig_index'].values, test_size=0.2, random_state=42, stratify=y
)

class F1RaceSet(Dataset):
    def __init__(self, X_cat, X_num, y):
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X_cat[idx], self.X_num[idx], self.y[idx]

class F1DNN(nn.Module):
    def __init__(self, cat_dims, num_num_features, embedding_dim=8, hidden_dim=128, num_classes=None):
        super().__init__()
        self.emb_layers = nn.ModuleList([
            nn.Embedding(cat_dim, embedding_dim) for cat_dim in cat_dims
        ])
        input_dim = embedding_dim * len(cat_dims) + num_num_features
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    def forward(self, x_cat, x_num):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.emb_layers)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.mlp(x)

batch_size = 128
trainset = F1RaceSet(X_cat_train, X_num_train, y_train)
testset = F1RaceSet(X_cat_test, X_num_test, y_test)
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(testset, batch_size=batch_size)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cat_dims = [int(df[col].max() + 1) for col in cat_cols]

model = F1DNN(cat_dims, len(num_cols), embedding_dim=8, hidden_dim=128, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 100

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_cat_batch, X_num_batch, y_batch in train_loader:
        X_cat_batch, X_num_batch, y_batch = X_cat_batch.to(device), X_num_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_cat_batch, X_num_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
    avg_loss = total_loss / len(trainset)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for X_cat_batch, X_num_batch, y_batch in test_loader:
        X_cat_batch, X_num_batch = X_cat_batch.to(device), X_num_batch.to(device)
        logits = model(X_cat_batch, X_num_batch)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(y_batch.numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

unique_y = np.unique(all_labels)
target_names = [f"{idx}: {name}" for idx, name in zip(unique_y, le_winner.inverse_transform(unique_y))]
print(classification_report(
    all_labels, all_preds,
    labels=unique_y,
    target_names=target_names,
    zero_division=0
))

torch.save(model.state_dict(), 'f1_dnn_embedding.pth')


過濾前總樣本數： 1110
只出現過一次的車手數量： 36
保留後樣本數： 1074
y範圍: 0 78 num_classes: 79
Epoch 1/100, Loss: 4.3222
Epoch 2/100, Loss: 3.7681
Epoch 3/100, Loss: 3.3702
Epoch 4/100, Loss: 3.0512
Epoch 5/100, Loss: 2.7772
Epoch 6/100, Loss: 2.5496
Epoch 7/100, Loss: 2.3170
Epoch 8/100, Loss: 2.1657
Epoch 9/100, Loss: 1.9906
Epoch 10/100, Loss: 1.8746
Epoch 11/100, Loss: 1.7475
Epoch 12/100, Loss: 1.5940
Epoch 13/100, Loss: 1.4889
Epoch 14/100, Loss: 1.3776
Epoch 15/100, Loss: 1.3018
Epoch 16/100, Loss: 1.2154
Epoch 17/100, Loss: 1.1259
Epoch 18/100, Loss: 1.0695
Epoch 19/100, Loss: 1.0140
Epoch 20/100, Loss: 0.9736
Epoch 21/100, Loss: 0.9037
Epoch 22/100, Loss: 0.8743
Epoch 23/100, Loss: 0.8238
Epoch 24/100, Loss: 0.7769
Epoch 25/100, Loss: 0.7495
Epoch 26/100, Loss: 0.7150
Epoch 27/100, Loss: 0.6912
Epoch 28/100, Loss: 0.6461
Epoch 29/100, Loss: 0.6110
Epoch 30/100, Loss: 0.5760
Epoch 31/100, Loss: 0.5721
Epoch 32/100, Loss: 0.5287
Epoch 33/100, Loss: 0.5202
Epoch 34/100, Loss: 0.4900
Epoch 35/100, Loss: 0.

In [ ]:
import gradio as gr
from collections import defaultdict

winners_df = winner
drivers_df = pd.read_csv('./data/drivers_updated.csv')

year_choices = sorted([int(x) for x in winners_df['year'].dropna().unique()])
grand_prix_choices = sorted([str(x) for x in winners_df['Grand Prix'].dropna().unique()])

max_data_year = max(year_choices)
recent_years = [max_data_year, max_data_year-1]
active_driver_set = set(drivers_df[drivers_df['year'].isin(recent_years)]['Driver'].astype(str).str.strip())

def get_all_station_drivers(grand_prix):
    df = winners_df[winners_df['Grand Prix'] == grand_prix]
    combos = []
    for _, row in df.iterrows():
        driver = str(row['Winner']).strip()
        car = str(row['Car']).strip()
        team = str(row['Car']).strip()  # 若有 Team 欄位請改用
        nationality = drivers_df[
            (drivers_df['Driver'] == driver) & (drivers_df['year'] == int(row['year']))
        ]['Nationality']
        nationality = nationality.values[0].strip() if not nationality.empty else "UNK"
        combos.append(dict(driver=driver, car=car, team=team, nationality=nationality, year=int(row['year'])))
    unique_combos = {}
    for c in combos:
        k = (c['driver'], c['car'], c['team'], c['nationality'])
        if k in unique_combos:
            unique_combos[k]['all_years'].append(c['year'])
        else:
            c['all_years'] = [c['year']]
            unique_combos[k] = c
    return list(unique_combos.values())

def predict_winner(year, grand_prix):
    year = int(year)
    # 新增判斷2024年也視為未來年份
    is_future = (year > max_data_year) or (year == 2024)
    combos = get_all_station_drivers(grand_prix)
    if not combos:
        return "查無該場比賽資料，無法預測。"

    probs_map = defaultdict(list)
    skipped_drivers = []
    for combo in combos:
        driver_name = combo['driver'].strip()
        if driver_name not in [x.strip() for x in le_winner.classes_]:
            skipped_drivers.append(driver_name)
            continue
        cat_inputs = [
            encoders['Car'].transform([combo['car'].strip()])[0] if combo['car'].strip() in encoders['Car'].classes_ else 0,
            encoders['Grand Prix'].transform([grand_prix.strip()])[0] if grand_prix.strip() in encoders['Grand Prix'].classes_ else 0,
            encoders['Nationality'].transform([combo['nationality'].strip()])[0] if combo['nationality'].strip() in encoders['Nationality'].classes_ else 0,
            encoders['Team'].transform([combo['team'].strip()])[0] if combo['team'].strip() in encoders['Team'].classes_ else 0
        ]
        cat_inputs = np.array(cat_inputs).reshape(1, -1)
        num_inputs = [0] * len(num_cols)
        if 'year' in num_cols:
            idx = num_cols.index('year')
            num_inputs[idx] = year
        num_inputs = scaler.transform([num_inputs])
        cat_tensor = torch.tensor(cat_inputs, dtype=torch.long).to(device)
        num_tensor = torch.tensor(num_inputs, dtype=torch.float32).to(device)
        model.eval()
        with torch.no_grad():
            logits = model(cat_tensor, num_tensor)
            prob = torch.softmax(logits, dim=1).cpu().numpy()[0]
            driver_idx = list(le_winner.classes_).index(driver_name)
            probs_map[driver_name].append((prob[driver_idx], combo))

    unique_probs = []
    for driver, prob_combos in probs_map.items():
        retired = is_future and (driver not in set(x.strip() for x in active_driver_set))
        best_prob, best_combo = max(prob_combos, key=lambda x: x[0])
        display_driver = driver + ("（退役）" if retired else "")
        unique_probs.append((best_prob, display_driver))
    unique_probs.sort(reverse=True)

    result_lines = []
    if unique_probs:
        best_driver = unique_probs[0][1]
        prob_str = "\n".join([f"{d}: {p*100:.2f}%" for p, d in unique_probs[:10]])
        year_desc = "未來預測" if is_future else f"{year}年"
        result_lines.append(f"{year_desc} {grand_prix} 預測最有可能奪冠：{best_driver}\n\n[依機率排序前10名]\n{prob_str}")
    else:
        result_lines.append("無任何模型學過的車手可預測。")
    if skipped_drivers:
        skipped_drivers = list(set(skipped_drivers))
        result_lines.append("\n---\n下列車手未納入訓練資料，無法預測：\n" + "、".join(skipped_drivers))
    return "\n".join(result_lines)

with gr.Blocks() as demo:
    year_in = gr.Dropdown(choices=year_choices + [2024, max(year_choices)+1], value=year_choices[-1], label="Year (年份，可選未來)")
    grand_prix_in = gr.Dropdown(choices=grand_prix_choices, value=grand_prix_choices[0], label="Grand Prix (場地/分站)")
    out_box = gr.Textbox(label="預測結果")
    btn = gr.Button("預測")
    btn.click(
        predict_winner,
        inputs=[year_in, grand_prix_in],
        outputs=out_box
    )
demo.launch()


* Running on local URL:  http://127.0.0.1:7892
* To create a public link, set `share=True` in `launch()`.


["'Alain  Prost '", "'Alan Jones '", "'Alberto  Ascari '", "'Ayrton  Senna '", "'Bill Vukovich'", "'Bruce  McLaren '", "'Carlos  Reutemann '", "'Carlos  Sainz '", "'Charles  Leclerc '", "'Clay  Regazzoni '", "'Damon  Hill '", "'Dan  Gurney '", "'Daniel  Ricciardo '", "'David  Coulthard '", "'Denny  Hulme '", "'Didier  Pironi '", "'Eddie  Irvine '", "'Elio  de Angelis '", "'Emerson  Fittipaldi '", "'Felipe  Massa '", "'Fernando  Alonso '", "'Gerhard  Berger '", "'Giancarlo  Fisichella '", "'Gilles  Villeneuve '", "'Graham  Hill '", "'Heinz-Harald  Frentzen '", "'Jack  Brabham '", "'Jackie  Stewart '", "'Jacky  Ickx '", "'Jacques  Laffite '", "'Jacques  Villeneuve '", "'James  Hunt '", "'Jean-Pierre  Jabouille '", "'Jenson  Button '", "'Jim  Clark '", "'Jo  Siffert '", "'Jochen  Rindt '", "'Jody  Scheckter '", "'John  Surtees '", "'John  Watson '", "'Johnny  Herbert '", "'Jose Froilan  Gonzalez '", "'Juan Manuel  Fangio '", "'Juan Pablo  Montoya '", "'Keke  Rosberg '", "'Kimi  RÃ¤ikkÃ¶ne